In [20]:
import sys
#!{sys.executable} -m pip install scikit-learn

In [21]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
from pathlib import Path
import gensim
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import numpy as np
import os
import matplotlib.pyplot as plt

In [22]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
# Initialize stemmer/lemmatizer (run once)
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Load the speeches

In [23]:
base_path_alvaro = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Project")

base_path = base_path_alvaro # change according to user

In [24]:
final_df = pd.read_csv(base_path / "Final_df.csv")
final_df = final_df.drop(columns=["Speech", "number_sentences", "number_tokens","matched_climate_keywords", "speeches_for_keyword_search"])

In [25]:
# Climate keyword according to the literature
climate_keywords = ["climate change", "global warming", "global warm","cap and trade", "paris accord", "emissions trading", "global average temperature", 
                      "kyoto protocol", "changing climate" ,"climate resilience","climate decay", "carbon dioxide", "carbon-dioxide","climate politics", 
                      "framework convention climate change", "bali roadmap", "bali action plan", 
                      "greenhouse gas", "greenhouse-gas","greenhouse effect", "climate mitigation", "climate action", "emissions", "temperature", "extreme weather", 
                      "global environmental change", "global environment", "global environmental" ,"climate variability", "low carbon", "renewable energy", 
                      "carbon emission", "climate pollutant", "climate pollutants", "carbon tax", "carbon footprint", "carbon neutrality", "net-zero", 
                      "net zero","net-zero","climate crisis", "climate summit", "climate catastrophe", "climate justice", "climate emergency", "climate funding",
                      "climate fund", "climate financing", "climate finance","climate peace", "climate agreement", "climate security", "climate ambition", 
                      "climate issue", "climate impact", "climate conference", "climate event", "climate challenge", "climate trust", "climate negotiation", 
                      "climate catastrophe", "climate risk", "climate goal", "climate change-related", "climate regime", "climate resilient", "climate policy", 
                      "carbon market", "carbon sink", "green climate", "green economy", "emission reduction", "emissions reduction", "carbon neutral", 
                      "ozone layer", "emissions trading scheme", "unfccc","ghg", "ghge", "co2", "co2 emission", "ipcc", "decarbonisation", "decarbonization"]

In [26]:
climate_keywords_dict = {
    # 1. Climate Science & Impacts
    "science_impacts": [
        "climate change", "global warm", "global warming", "global average temperature",
        "climate variability", "extreme weather", "climate impact",
        "greenhouse effect", "temperature", "climate catastrophe",
        "climate risk", "ozone layer", "global environmental change",
        "global environmental", "global environment", 
        "climate change-related", "climate risk", "changing climate",
        "climate decay"
    ],

    # 2. Policy & Agreements
    "policy_agreements": [
        "paris accord", "kyoto protocol", "unfccc", "climate policy",
        "framework convention on climate change", "climate agreement",
        "climate regime", "climate negotiation", "climate ambition",
        "climate security", "bali roadmap", "ipcc", "bali action plan"
    ],

    # 3. Carbon & Emissions
    "carbon_emissions": [
        "carbon dioxide", "co2", "carbon emission", "carbon tax",
        "carbon footprint", "carbon neutrality", "carbon neutral",
        "carbon market", "carbon sink", "low carbon", "emissions",
        "emission reduction", "emissions reduction", "ghg", "ghge",
        "climate pollutant", "climate pollutants", "carbon neutral",
        "greenhouse gas", "decarbonisation", "decarbonization", 
        "greenhouse-gas"
    ],

    # 4. Climate Action & Solutions
    "action_solutions": [
        "climate action", "climate mitigation", "renewable energy",
        "climate resilience", "climate resilient", "green economy",
        "climate funding", "climate fund", "climate financing", "climate finance",
        "net-zero", "net zero", "green climate", "cap and trade",
        "emissions trading scheme", "emissions trading"
    ],

    # 5. Sociopolitical Climate Issues
    "sociopolitical": [
        "climate justice", "climate emergency", "climate crisis",
        "climate summit", "climate politics", "climate peace",
        "climate challenge", "climate goal", "climate issue",
        "climate event", "climate conference", "climate trust"
    ]
}
    

# Join all the bigrams acordingly

In [27]:
bigrams = ["climate change", "global warming", "global warm","cap and trade", "paris accord", "emissions trading", "global average temperature", 
            "kyoto protocol", "changing climate" ,"climate resilience","climate decay", "carbon dioxide", "carbon-dioxide","climate politics", 
            "framework convention climate change", "bali roadmap", "bali action plan", 
            "greenhouse gas", "greenhouse-gas","greenhouse effect", "climate mitigation", "climate action", "emissions", "temperature", "extreme weather", 
            "global environmental change", "global environment", "global environmental" ,"climate variability", "low carbon", "renewable energy", 
            "carbon emission", "climate pollutant", "climate pollutants", "carbon tax", "carbon footprint", "carbon neutrality", 
            "net zero","net-zero","climate crisis", "climate summit", "climate catastrophe", "climate justice", "climate emergency", "climate funding",
            "climate fund", "climate financing", "climate finance","climate peace", "climate agreement", "climate security", "climate ambition", 
            "climate issue", "climate impact", "climate conference", "climate event", "climate challenge", "climate trust", "climate negotiation", 
            "climate catastrophe", "climate risk", "climate goal", "climate change-related", "climate regime", "climate resilient", "climate policy", 
            "carbon market", "carbon sink", "green climate", "green economy", "emission reduction", "emissions reduction", "carbon neutral", 
            "ozone layer", "emissions trading scheme", "co2 emission"]

In [28]:
bigrams_1 = sorted(bigrams, key=lambda x: -len(x.split()))

In [29]:
def replace_bigrams(text, bigrams):
    for phrase in bigrams:
        # Normalize the phrase: convert dashes to spaces
        normalized_phrase = phrase.replace("-", " ")
        # Build regex pattern from normalized phrase
        pattern = r'\b' + re.escape(normalized_phrase) + r'\b'
        # Replace spaces (now consistent) with underscores
        underscored = normalized_phrase.replace(" ", "_")
        text = re.sub(pattern, underscored, text)
    return text

In [30]:
tqdm.pandas()  # Enable progress_apply in pandas

final_df["cleaned_speeches_postagging_expanded"] = final_df["cleaned_speeches_postagging_expanded"].progress_apply(
    lambda x: replace_bigrams(x, bigrams)
)

100%|██████████| 6457/6457 [00:50<00:00, 128.88it/s]


In [31]:
from gensim import corpora
from gensim.models import LdaModel
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

In [62]:
def compute_coherence_and_perplexity(train_corpus, test_corpus, dictionary, k, a, b):
    lda_model = LdaModel(
        corpus=train_corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=10,
        alpha=a,
        eta=b,
        minimum_probability=0.1
    )

    # Perplexity on held-out corpus
    perplexity = lda_model.log_perplexity(test_corpus)

    return perplexity, lda_model

In [63]:
def gridsearch_lda(train_corpus, test_corpus, max_topics_range, max_alpha, max_eta, dictionary):
    from tqdm.auto import tqdm
    import numpy as np

    # Clear previous progress bars (critical for multiple runs)
    if 'tqdm' in globals():
        tqdm._instances.clear()

    topics_range = range(2, max_topics_range + 1)
    alpha = list(np.linspace(0.01, max_alpha, 3))
    eta = list(np.linspace(0.01, max_eta, 3))

    model_results = {
        'Topics': [], 'Alpha': [], 'Eta': [],
        'Perplexity': [],
        # 'Model': []  # Consider not storing all models to save memory
    }
    #import tqdm
    if 1 == 1:
        pbar = tqdm(total=len(eta) * len(alpha) * len(topics_range), desc="Grid Search", leave=False)
        for k in topics_range:
            for a in alpha:
                for b in eta:
                    try:
                        log_perplexity, model = compute_coherence_and_perplexity(
                            train_corpus, test_corpus, dictionary, k, a, b)
                        
                        model_results['Topics'].append(k)
                        model_results['Alpha'].append(a)
                        model_results['Eta'].append(b)
                        model_results['Perplexity'].append(np.exp2(-log_perplexity))  # Convert to perplexity
                    except Exception as e:
                        print(f"Failed for k={k}, alpha={a}, beta={b}: {e}")
                        # Append None for failed cases
                        model_results['Topics'].append(k)
                        model_results['Alpha'].append(a)
                        model_results['Eta'].append(b)
                        model_results['Perplexity'].append(None)
                    finally:
                        pbar.update(1)

    return model_results

In [64]:
def plot_grouped_final_hyperparaemters_perplexity(df, period_label, i, climate):
    plt.figure(figsize=(12, 7))
    for (alpha, eta), group in df:
        group_sorted = group.sort_values(by="Topics")
        label = f"α={alpha}, η={eta}"
        plt.plot(group_sorted["Topics"], group_sorted["Perplexity"], marker='o', label=label)

    plt.xlabel("Number of Topics (k)")
    plt.ylabel("Perplexity")
    if climate:
        plt.title(f"Perplexity vs Number of Topics for each (Alpha, Eta) combination — Income {i} (Climate Speeches {period_label})")
        filename = f"perplexity_income_{i}_climate_{period_label}.png"
    else:
        plt.title(f"Perplexity vs Number of Topics for each (Alpha, Eta) combination — Income {i} (Not Climate Speeches {period_label})")
        filename = f"perplexity_income_{i}_non_climate_{period_label}.png"

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.grid(True)
    output_dir = r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Plot saved to: {save_path}")


In [65]:
def plot_topic_words(lda_model, topic_id, i, period_label, climate, n_words=10):
    """
    Plots a bar chart of the top words in a topic with their probabilities.
    The color of the bars depends on the topic ID.
    """
    import matplotlib.pyplot as plt

    # Define custom colors for first few topics
    color_map = {
        0: 'red',
        1: 'blue',
        2: 'green',
        3: 'black',
        4: 'orange',
        5: 'purple',
        6: 'brown',
        7: 'pink',
        8: 'cyan',
        9: 'olive',
        10: 'gray', 
        11: 'yellow'
    }
    
    bar_color = color_map.get(topic_id, 'gray')  # Default to gray if not in map

    # Get the topic-word distribution
    topic_words = lda_model.show_topic(topic_id, topn=n_words)
    words = [word for word, prob in topic_words]
    probs = [prob for word, prob in topic_words]

    # Create plot
    plt.figure(figsize=(10, 5))
    plt.barh(words, probs, color=bar_color)

    if climate:
        plt.title(f"Topic {topic_id} — Top {n_words} Words — Income {i} (Climate Speeches {period_label})", fontsize=14)
        filename = f"top_words_topic_{topic_id}_income_{i}_climate_{period_label}.png"
    else:
        plt.title(f"Topic {topic_id} — Top {n_words} Words — Income {i} (Not Climate Speeches {period_label})", fontsize=14)
        filename = f"top_words_topic_{topic_id}_income_{i}_climate_{period_label}.png"

    plt.xlabel("Probability", fontsize=12)
    plt.gca().invert_yaxis()  # Highest probability at the top
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.tight_layout()
    output_dir = r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Plot saved to: {save_path}")
    

In [66]:
final_df

,Session,Year,ISO-Code,Income Level,cleaned_speeches_postagging_expanded,contains_climate_keyword
0,45,1990,AFG,1,allow first sir congratulate unanimous electio...,False
1,45,1990,AGO,2,first would like congratulate sir election pre...,False
2,45,1990,ALB,2,special pleasure speak session general assembl...,False
3,45,1990,ARE,4,president behalf delegation united arab emirat...,False
4,45,1990,ARG,2,president general assembly fortyfifth session ...,False
...,...,...,...,...,...,...
6452,79,2024,WSM,2,excellency extend congratulation excellency ph...,True
6453,79,2024,YEM,1,lady gentleman happy coincidence address today...,True
6454,79,2024,ZAF,3,president session general assembly philemon ya...,True
6455,79,2024,ZMB,2,lady gentleman congratulate excellency assumpt...,True


In [69]:
def LDA_for_big_data_paris_agreement(df, climate_only = True):

    paris_periods = {
        "1990_2014": (1990, 2014),
        "2015_2024": (2015, 2024)
    }

    for period_label, (start_year, end_year) in paris_periods.items():
        period_df = df[(df["Year"] >= start_year) & (df["Year"] <= end_year)].copy()
        print(f"\n=== Period: {period_label} ===")
        

        for i in range(1, 5):  # Assuming Income Label 1 to 4
            income_df = period_df[period_df["Income Level"] == i].copy()
            print(f">>> Income Level {i}: {len(income_df)} documents")

            if climate_only:
                income_df_climate = income_df[income_df['contains_climate_keyword'] == climate_only]
                print(f">>> Income Level {i} for climate sentence is {climate_only}: {len(income_df_climate)} documents")
            else:
                climate_only = False
                income_df_climate = income_df[income_df['contains_climate_keyword'] == climate_only]
                print(f">>> Income Level {i} for climate sentence is {climate_only}: {len(income_df_climate)} documents")

            ##################################################### LDAs model ################################################################
            income_df_climate = income_df_climate.copy() # copy otehrwise gives an error

            income_df_climate["tokens"] = income_df_climate["cleaned_speeches_postagging_expanded"].str.split() # tokenize accordingly 
            
            texts = income_df_climate["tokens"].dropna().tolist() # drop na values of the tokenized
            
            train_docs, test_docs = train_test_split(texts, test_size=0.1, random_state=42) # divide between train and test 
            
            dictionary = corpora.Dictionary(texts) # create the dicitonary
            
            train_corpus = [dictionary.doc2bow(doc) for doc in train_docs] # train set
            
            test_corpus = [dictionary.doc2bow(doc) for doc in test_docs] # test set

            # Get sorted list of word:ID pairs
            results = gridsearch_lda(
                train_corpus=train_corpus,
                test_corpus=test_corpus,
                max_topics_range=10,
                max_alpha=1,
                max_eta=1,
                dictionary=dictionary
            )

            # Convert to DataFrame and sort by coherence
            results_df = pd.DataFrame({k: v for k, v in results.items() if k != 'Model'})
            # Group results by (Alpha, Beta)
            grouped = results_df.groupby(["Alpha", "Eta"])

            ############## Plot hyperparameters ################################
            plot_grouped_final_hyperparaemters_perplexity(grouped, period_label, i, climate = climate_only)

            number_of_topics_optimized = results_df.sort_values(by='Perplexity', ascending=True).iloc[0]['Topics']
            alpha_optimized = results_df.sort_values(by='Perplexity', ascending=True).iloc[0]['Alpha']
            eta_optimized = results_df.sort_values(by='Perplexity', ascending=True).iloc[0]['Eta']
            
            print("The best performing model has", number_of_topics_optimized, "and an alpha of", alpha_optimized, "and eta of", eta_optimized)

            final_lda = LdaModel(
                corpus=train_corpus + test_corpus,  # Combined train + test data
                id2word=dictionary,
                num_topics=number_of_topics_optimized,       # From your grid search
                alpha=alpha_optimized,          # From your grid search
                eta=eta_optimized,             # From your grid search (eta = Beta)
                passes=10,           # Increase for better convergence
                random_state=42
            )


            # Plot for each topic
            for topic_id in range(final_lda.num_topics):
                plot_topic_words(final_lda, topic_id, i, period_label, climate = climate_only)

    return results_df, 
    

LDA_for_big_data_paris_agreement(final_df, climate_only = True)


=== Period: 1990_2014 ===
>>> Income Level 1: 1289 documents
>>> Income Level 1 for climate sentence is True: 346 documents


Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\perplexity_income_1_climate_1990_2014.png
The best performing model has 2.0 and an alpha of 0.01 and eta of 0.505
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_0_income_1_climate_1990_2014.png
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_1_income_1_climate_1990_2014.png
>>> Income Level 2: 1343 documents
>>> Income Level 2 for climate sentence is True: 504 documents


Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\perplexity_income_2_climate_1990_2014.png
The best performing model has 2.0 and an alpha of 1.0 and eta of 0.505
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_0_income_2_climate_1990_2014.png
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_1_income_2_climate_1990_2014.png
>>> Income Level 3: 900 documents
>>> Income Level 3 for climate sentence is True: 451 documents


Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\perplexity_income_3_climate_1990_2014.png
The best performing model has 2.0 and an alpha of 1.0 and eta of 0.505
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_0_income_3_climate_1990_2014.png
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_1_income_3_climate_1990_2014.png
>>> Income Level 4: 1012 documents
>>> Income Level 4 for climate sentence is True: 472 documents


Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\perplexity_income_4_climate_1990_2014.png
The best performing model has 2.0 and an alpha of 1.0 and eta of 0.505
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_0_income_4_climate_1990_2014.png
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_1_income_4_climate_1990_2014.png

=== Period: 2015_2024 ===
>>> Income Level 1: 281 documents
>>> Income Level 1 for climate sentence is True: 232 documents


Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\perplexity_income_1_climate_2015_2024.png
The best performing model has 4.0 and an alpha of 0.01 and eta of 1.0
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_0_income_1_climate_2015_2024.png
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_1_income_1_climate_2015_2024.png
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_2_income_1_climate_2015_2024.png
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_3_income_1_climate_2015_2024.png
>>> Income Level 2: 502 documents
>>> Income Level 2 for climate sentence is True: 421 documents


Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\perplexity_income_2_climate_2015_2024.png
The best performing model has 2.0 and an alpha of 1.0 and eta of 0.505
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_0_income_2_climate_2015_2024.png
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_1_income_2_climate_2015_2024.png
>>> Income Level 3: 540 documents
>>> Income Level 3 for climate sentence is True: 442 documents


Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\perplexity_income_3_climate_2015_2024.png
The best performing model has 2.0 and an alpha of 0.505 and eta of 0.505
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_0_income_3_climate_2015_2024.png
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_1_income_3_climate_2015_2024.png
>>> Income Level 4: 590 documents
>>> Income Level 4 for climate sentence is True: 512 documents


Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\perplexity_income_4_climate_2015_2024.png
The best performing model has 2.0 and an alpha of 1.0 and eta of 0.505
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_0_income_4_climate_2015_2024.png
Plot saved to: C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\top_words_topic_1_income_4_climate_2015_2024.png


(    Topics  Alpha    Eta   Perplexity
 0        2  0.010  0.010  1353.209034
 1        2  0.010  0.505   344.969984
 2        2  0.010  1.000   412.165546
 3        2  0.505  0.010  1350.911102
 4        2  0.505  0.505   344.343166
 ..     ...    ...    ...          ...
 76      10  0.505  0.505   534.568744
 77      10  0.505  1.000   486.933693
 78      10  1.000  0.010  5043.657261
 79      10  1.000  0.505   548.970551
 80      10  1.000  1.000   497.739073
 
 [81 rows x 4 columns],)

In [70]:
LDA_for_big_data_paris_agreement(final_df, climate_only = False)


=== Period: 1990_2014 ===
>>> Income Level 1: 1289 documents
>>> Income Level 1 for climate sentence is False: 943 documents


Grid Search:   5%|▍         | 4/81 [01:26<26:49, 20.91s/it]

KeyboardInterrupt: 